# Image Transformations with R
## Skeleton Notebook for Practice (R version)

**Goal**: Learn to represent, transform, and analyze images using R matrices and linear algebra. This is the R translation of the expanded NumPy project.

📌 **How to use**: Read instructions, fill `# TODO` code cells, then compare with the Solution notebook.

**Required packages** (run once):
```r
install.packages(c("ggplot2", "reshape2"))   # reshape2 optional but helpful
```

## Project Workflow Flowchart

![Workflow Flowchart](image_transform_workflow_flowchart.png)

The flowchart above shows the complete learning journey (same as Python version). Study it before starting the exercises.

## Learning Objectives
By completing this notebook you will be able to:
1. Represent images as R matrices and understand pixel-value mapping.
2. Perform basic and advanced image transformations (invert, flip, rotate, brightness, noise, blur).
3. Use linear algebra (`solve()`) to recover or transform image data.
4. Create parameter-driven simulations and measure impact with MAE, RMSE, PSNR.
5. Consider **audience data literacy** when explaining visualizations.
6. Apply these skills to practical data pipeline / ML preprocessing tasks in R.

## 1. Setup & Helper Function

We will create a nice `show_image()` helper using `ggplot2::geom_tile()` because it gives clean, labeled pixel plots that are excellent for teaching.

In [ ]:
# TODO: Load libraries
library(ggplot2)
# library(reshape2)   # uncomment if you want melt() for long format

print("Libraries loaded.")

# TODO: Create a nice show_image helper using ggplot2
show_image <- function(mat, title = "Image", low = "black", high = "white") {
  # Convert matrix to long data frame
  df <- data.frame(
    row = rep(seq_len(nrow(mat)), each = ncol(mat)),
    col = rep(seq_len(ncol(mat)), times = nrow(mat)),
    value = as.vector(t(mat))   # t() because ggplot uses row-major visually
  )
  p <- ggplot(df, aes(x = col, y = -row, fill = value)) +
    geom_tile(color = "gray30", linewidth = 0.3) +
    scale_fill_gradient(low = low, high = high, na.value = "gray") +
    coord_fixed() +
    labs(title = title) +
    theme_minimal() +
    theme(
      axis.title = element_blank(),
      axis.text = element_blank(),
      axis.ticks = element_blank(),
      panel.grid = element_blank(),
      plot.title = element_text(hjust = 0.5, face = "bold")
    )
  print(p)
  # Also print summary stats
  cat(sprintf("[%s] dim = %s, class = %s, min = %.2f, max = %.2f, mean = %.2f\n",
              title, paste(dim(mat), collapse = " x "), class(mat)[1],
              min(mat), max(mat), mean(mat)))
  invisible(mat)
}

### Define the Base Heart Image

Same 7×7 matrix as the Python version.

In [ ]:
# TODO: Define the heart matrix
heart_img <- matrix(c(
  255,   0,   0, 255,   0,   0, 255,
    0, 127.5, 127.5,   0, 127.5, 127.5,   0,
    0, 127.5, 127.5, 127.5, 127.5, 127.5,   0,
    0, 127.5, 127.5, 127.5, 127.5, 127.5,   0,
  255,   0, 127.5, 127.5, 127.5,   0, 255,
  255, 255,   0, 127.5,   0, 255, 255,
  255, 255, 255,   0, 255, 255, 255
), nrow = 7, byrow = TRUE)

show_image(heart_img, "Original Heart Image")

## 2. Understanding the Image as an R Matrix

**Tasks**: Inspect dimensions, class, specific pixel values, and understand why the heart shape appears.

In [ ]:
# TODO: Inspect the matrix
cat("dim(heart_img)    :", dim(heart_img), "\n")
cat("class(heart_img)  :", class(heart_img), "\n")
cat("Center (4,4)      :", heart_img[4, 4], "\n")   # R is 1-indexed!
cat("Top row           :", heart_img[1, ], "\n")
cat("Bottom-right 2x2  :\n"); print(heart_img[6:7, 6:7])

## 3. Basic Image Transformations

### 3.1 Invert Colors — Multiple Ways in R

In [ ]:
# TODO: Invert using simple subtraction (recommended)
inverted1 <- 255 - heart_img
show_image(inverted1, "Inverted Heart (255 - img)")

# TODO: Alternate 2: max - img
inverted2 <- max(heart_img) - heart_img
show_image(inverted2, "Inverted Heart (max - img)")

# TODO: Alternate 3: 255 - img but force integer if needed
inverted3 <- 255L - as.integer(heart_img)
show_image(inverted3, "Inverted Heart (integer version)")

### 3.2 Flip and Rotate

In [ ]:
# TODO: Geometric transforms
flipped_h <- heart_img[, ncol(heart_img):1]          # horizontal flip
flipped_v <- heart_img[nrow(heart_img):1, ]          # vertical flip
transposed <- t(heart_img)                           # transpose
# 90 degree clockwise rotation
rotated_90 <- t(heart_img[nrow(heart_img):1, ])

show_image(flipped_h, "Flipped Horizontal")
show_image(transposed, "Transposed (original 'Rotate')")

## 4. Intensity Adjustments & Adding Noise

In [ ]:
# TODO: Brightness
brightness_factor <- 1.4
bright_img <- pmin(pmax(heart_img * brightness_factor, 0), 255)
show_image(bright_img, paste0("Brightness x", brightness_factor))

# TODO: Gaussian noise
set.seed(42)
noise_sd <- 25
noise <- matrix(rnorm(length(heart_img), mean = 0, sd = noise_sd), nrow = 7)
noisy_img <- pmin(pmax(heart_img + noise, 0), 255)
show_image(noisy_img, paste0("Noisy (sd = ", noise_sd, ")"))

## 5. Advanced: Simple Gaussian Blur (Base R)

We implement a lightweight 2D Gaussian blur using `outer()` + `dnorm`. For production use consider the `imager` package.

In [ ]:
# Simple Gaussian blur function (base R)
gaussian_blur <- function(img, sigma = 1) {
  ksize <- ceiling(3 * sigma)
  x <- seq(-ksize, ksize)
  kernel <- outer(x, x, function(a, b) dnorm(a, sd = sigma) * dnorm(b, sd = sigma))
  kernel <- kernel / sum(kernel)
  # Pad and convolve (simple version)
  padded <- rbind(
    matrix(rep(img[1, ], ksize), nrow = ksize, byrow = TRUE),
    img,
    matrix(rep(img[nrow(img), ], ksize), nrow = ksize, byrow = TRUE)
  )
  padded <- cbind(
    matrix(rep(padded[, 1], ksize), ncol = ksize),
    padded,
    matrix(rep(padded[, ncol(padded)], ksize), ncol = ksize)
  )
  # Convolution via filter (or manual)
  blurred <- img * 0
  for (i in 1:nrow(img)) {
    for (j in 1:ncol(img)) {
      region <- padded[(i):(i + 2*ksize), (j):(j + 2*ksize)]
      blurred[i, j] <- sum(region * kernel)
    }
  }
  return(blurred)
}

# TODO: Try different sigma
blurred <- gaussian_blur(heart_img, sigma = 0.8)
show_image(blurred, "Gaussian Blur (sigma = 0.8)")

## 6. Linear Algebra: Solving for the Image

Generate random matrix `A` and solve `A %*% X = heart_img`.

In [ ]:
# TODO: Linear algebra recovery
set.seed(123)
A <- matrix(sample(10:240, 49, replace = TRUE), nrow = 7)
show_image(A, "Random Matrix A")

X <- solve(A, heart_img)                 # direct equivalent of np.linalg.solve
show_image(X, "Solved coefficient matrix X")

reconstructed <- A %*% X
show_image(reconstructed, "Reconstructed (A %*% X)")
cat("Reconstruction error (Frobenius):
    ", sqrt(sum((reconstructed - heart_img)^2)), "\n")

## 7. Simulation Section — Change Parameters & Observe Impact

**Core new feature**. Edit the parameters at the top of the cell and re-run to see different results + metrics.

In [ ]:
# ========== R SIMULATION CELL - EDIT THESE VALUES ==========
brightness   <- 0.75
noise_sd     <- 30
blur_sigma   <- 0.9
do_invert    <- FALSE
flip_mode    <- "none"     # "h", "v", or "none"

# Apply pipeline
sim <- heart_img
sim <- pmin(pmax(sim * brightness, 0), 255)
if (noise_sd > 0) {
  set.seed(42)
  sim <- sim + matrix(rnorm(length(sim), 0, noise_sd), nrow = nrow(sim))
  sim <- pmin(pmax(sim, 0), 255)
}
if (blur_sigma > 0) {
  sim <- gaussian_blur(sim, sigma = blur_sigma)
}
if (do_invert) sim <- 255 - sim
if (flip_mode == "h") sim <- sim[, ncol(sim):1]
if (flip_mode == "v") sim <- sim[nrow(sim):1, ]

# Metrics
mae  <- mean(abs(sim - heart_img))
mse  <- mean((sim - heart_img)^2)
rmse <- sqrt(mse)
psnr <- if (rmse > 0) 20 * log10(255 / rmse) else Inf
cat("=== SIMULATION RESULTS (R) ===\n")
cat(sprintf("brightness = %.2f | noise_sd = %.1f | blur_sigma = %.2f\n", brightness, noise_sd, blur_sigma))
cat(sprintf("MAE  = %.2f\n", mae))
cat(sprintf("RMSE = %.2f\n", rmse))
cat(sprintf("PSNR = %.2f dB\n", psnr))

# Side-by-side plot
par(mfrow = c(1, 2))
show_image(heart_img, "Original")
show_image(sim, paste0("Transformed (MAE=", round(mae,1), ")"))
par(mfrow = c(1, 1))

## 8. More Practice Challenges

Complete at least two:
1. Create your own 7×7 or 9×9 shape (plus sign, star, letter).
2. Chain several transforms and describe the visual effect.
3. Add strong noise then recover with your blur function. Compare MAE before vs after.
4. Write a short explanation of one transform suitable for a non-technical audience.

In [ ]:
# TODO: Practice - create a plus sign
plus <- matrix(0, nrow = 7, ncol = 7)
plus[4, ] <- 200
plus[, 4] <- 200
show_image(plus, "Plus Sign Shape")

# Apply brightness + noise + blur
chained <- pmin(pmax(plus * 0.8 + matrix(rnorm(49,0,15),7), 0), 255)
chained <- gaussian_blur(chained, sigma = 0.6)
show_image(chained, "Chained transforms on plus sign")

## Conclusion & Key Takeaways (R version)

- In R we use matrices + `solve()`, `t()`, vectorized arithmetic — very powerful and often more concise than Python/NumPy for linear algebra.
- Multiple ways exist to achieve the same visual result — choose the clearest one.
- Simulation + metrics turn subjective "looks good" into objective engineering.
- **Audience matters**: Match technical depth to the reader's data literacy.

---
**Great work completing the R skeleton!** Open the Solution notebook to see polished code, alternate methods, and full explanations.